In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDaC/notebooks


## Env

In [2]:
from src.utils.env import prepare_environment

prepare_environment("../api_keys")

INFO 08-12 19:23:16 [src.utils.env:30] Setting API tokens: ['WANDB_API_KEY', 'OPENPIPE_API_KEY', 'OPENAI_API_KEY']
INFO 08-12 19:23:16 [src.utils.env:42] Setting additional variables: {'NCCL_CUMEM_ENABLE': '0', 'VLLM_USE_V1': '0', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn', 'ART_SERVER_TIMEOUT': '300'}


## Model

In [9]:
from src.configs import art_configs
from src.configs import vllm_configs
from src.models import PathConfig

print("Available ART model configurations:")
print(art_configs.available_configs())

print("Available vLLM model configurations:")
print(vllm_configs.available_configs())

base_model = "unsloth/Qwen3-14B"
project_name = "easy2hard_dac"

path_config = PathConfig(
    base_model=base_model,
    project_name=project_name,
)

Available ART model configurations:
['unsloth/Llama-4-Scout-17B-16E-Instruct', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Qwen2-7B', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen3-14B']
Available vLLM model configurations:
['unsloth/Llama-4-Scout-17B-16E-Instruct', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Qwen2-7B', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen3-14B']


In [10]:
import art

host = "0.0.0.0"
port = 8200

model = art.Model(
    name=base_model,
    project=path_config.project_name,
    inference_api_key=os.getenv("OPENAI_API_KEY", "default"),
    inference_base_url=f"https://{host}:{port}/v1",
)

## Data

In [11]:
from datasets import Dataset, load_dataset, DatasetDict

dataset_dict: DatasetDict = load_dataset(
    path="furonghuang-lab/Easy2Hard-Bench",
    name="E2H-AMC",
    split=None,
)  # type: ignore

train_data: Dataset = dataset_dict["train"]
test_data: Dataset = dataset_dict["eval"]

## Inference Clients

In [12]:
from src.vllm_client import VllmClient, VllmRouter

inference_clients: list[VllmClient] = []
vllm_server_ports = [8200]
for port in vllm_server_ports:
    inference_clients.append(VllmClient.from_connection(port=port, base_model=base_model))

vllm_router: VllmRouter = VllmRouter(inference_clients)

## Config 

In [25]:
from experiments.easy2hard.trainer import Easy2HardTrainer
from src.trainer import PromptConfig, StopCriteria

prompt_config = PromptConfig(
    system_root="dac_sys_prompt_gilad_v2_root",
    system_inter="dac_sys_prompt_gilad_v2_inter",
    system_leaf="dac_sys_prompt_gilad_v2_leaf",
)

# prompt_config = PromptConfig(
#     system_root="",
#     system_inter="",
#     system_leaf="",
# )

stop_criteria = StopCriteria(max_depth=2)

trainer = Easy2HardTrainer(
    model=model,
    vllm_router=vllm_router,
    path_config=path_config,
    prompt_config=prompt_config,
    stop_criteria=stop_criteria,
)

### Inference Test

In [27]:
import random

idx = random.randint(0, len(train_data) - 1)
# idx = 228
print(f"Selected random index: {idx}")
sample = train_data[idx]
problem = sample["problem"]
true_answer = sample["answer"]

print(f"Problem: {problem}")
print(f"Answer: {true_answer}")
print("-" * 200)
print()

preds = await trainer.predict([sample], verbose=True, seed=0, extra_body={"chat_template_kwargs": {"enable_thinking": False}})

Selected random index: 767
Problem: Makarla attended two meetings during her $9$-hour work day. The first meeting took $45$ minutes and the second meeting took twice as long. What percent of her work day was spent attending meetings?
Answer: 25
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------



predict:   0%|          | 0/1 [00:00<?, ?it/s]

Role: SYSTEM
Content: You are a highly capable and truthful AI assistant that excels at logical reasoning.

When encountering complex tasks, you may break them down into smaller, manageable sub-tasks. When you do so, these sub-tasks will be assigned to sub-agent to solve and the answer is then immediately reported back to you. 

There are strict formatting rules which you must follow:

Your Turn Options:
- You may reason, and then create a sub-task within <task> </task> block.
- You may reason, and then provide a final answer within <answer> </answer> block, which ends the conversation.

Sub-Task Requirements:
- Each sub-task must be fully self-contained, include all context, instructions, and expected output detail level. 
- Only the text between the <task> </task> marks is received by the sub-agent as input.
- The sub-agent does not retain any conversational history at all, so every sub-task must include the full context and information necessary, including any relevant prior answers

## Run Benchmark

### Results

- No Sys + No Inst + Depth 0 - 0.356
- No Sys + Inst + Depth 0 - 0.147
- Leaf Sys + No Inst + Depth 0 - 0.34
- Leaf Sys + Inst + Depth 0 - 0.319
- Full Sys + No Inst + Depth 1 - 0.32
- Full Sys + Inst + Depth 1 - 0.292

In [ ]:
groups = await trainer.rollout(
    test_data.to_list(),
    max_exceptions=0.025,
)

In [ ]:
from pprint import pprint
from src.utils.analysis import average_metrics

print("Global Metrics:")
all_trajectories = [tr for group in groups for tr in group.trajectories]
pprint(average_metrics(all_trajectories))

print("Answered Metrics:")
answered_trajectories = [tr for tr in all_trajectories if tr.metrics["gave_answer"] == 1]
pprint(average_metrics(answered_trajectories))

print("Unanswered Metrics:")
unanswered_trajectories = [tr for tr in all_trajectories if tr.metrics["gave_answer"] == 0]
pprint(average_metrics(unanswered_trajectories))

In [ ]:
from src.utils.analysis import to_dataframe

df = to_dataframe(all_trajectories)

display(df.describe())